# Sheepherding Simulation: Tables and Analysis


In [1]:
CSV_PATH = "../data/all_runs_merged_full.csv"

In [2]:
import pandas as pd
import numpy as np
from scipy import stats, optimize

# Load
df = pd.read_csv(CSV_PATH)
if df.columns[0].startswith('"BehaviorSpace') or df.columns[0].startswith('BehaviorSpace'):
    df = pd.read_csv(CSV_PATH, skiprows=6)

df['success'] = df['done?'].astype(str).str.lower() == 'true'
for col in ['mean-spreadness', 'ticks-to-success', 'lost-sheep', 'dogs-distance', 'sheep-captured-count']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

dogs = sorted(df['nb-dogs'].unique())
sheep = sorted(df['nb-sheep'].unique())
print(f"{len(df)} runs | D = {dogs} | N = {sheep} | {len(dogs)*len(sheep)} conditions")

11000 runs | D = [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(10), np.int64(15), np.int64(20), np.int64(25), np.int64(35)] | N = [np.int64(5), np.int64(10), np.int64(25), np.int64(50), np.int64(100), np.int64(150), np.int64(200), np.int64(250), np.int64(300), np.int64(350), np.int64(400)] | 110 conditions


## 1. Success Rates (%)

In [3]:
sr = df.groupby(['nb-sheep','nb-dogs'])['success'].mean().unstack() * 100

def color_sr(val):
    if pd.isna(val): return ''
    if val >= 90: return 'background-color: #c6efce'
    if val >= 15: return 'background-color: #ffeb9c'
    return 'background-color: #ffc7ce'

sr.style.map(color_sr).format('{:.0f}')

nb-dogs,1,2,3,4,6,10,15,20,25,35
nb-sheep,,,,,,,,,,
5,100,100,100,98,100,100,99,99,98,99
10,99,100,100,100,99,99,97,86,85,70
25,100,100,100,99,99,98,98,98,93,88
50,100,100,100,99,99,99,99,97,94,81
100,100,100,100,100,100,100,98,95,92,95
150,1,80,99,97,96,95,97,97,97,92
200,0,9,57,68,71,89,87,93,94,89
250,0,0,5,11,46,71,85,89,93,91
300,0,0,0,1,10,50,74,93,93,90


## 2. Mean Ticks to Success (successful runs)

In [4]:
tts = df[df['success']].groupby(['nb-sheep','nb-dogs'])['ticks-to-success'].mean().unstack()
tts.style.format('{:,.0f}', na_rep='---')

nb-dogs,1,2,3,4,6,10,15,20,25,35
nb-sheep,,,,,,,,,,
5,"1,966","1,556","1,398","1,386","1,386","1,564","2,120","2,458","2,665","2,660"
10,"1,565","1,642","1,593","1,606","1,693","2,117","2,538","3,173","3,193","4,101"
25,"1,619","1,916","1,783","1,820","1,984","1,941","2,594","3,139","2,800","3,490"
50,"2,780","2,618","2,219","2,355","2,444","2,399","2,625","2,906","2,685","3,159"
100,"3,125","2,353","2,905","2,835","3,198","3,177","2,946","3,360","3,177","3,405"
150,"8,976","5,137","3,833","3,948","4,486","3,554","3,701","3,241","3,627","3,385"
200,---,"8,134","6,274","5,328","5,244","4,346","3,876","3,732","3,366","3,050"
250,---,---,"8,912","6,197","5,821","5,196","4,476","3,980","4,188","4,023"
300,---,---,---,"7,848","6,464","5,598","4,851","4,483","4,485","3,752"


## 3. Mean Spreadness

In [5]:
ms = df.groupby(['nb-sheep','nb-dogs'])['mean-spreadness'].mean().unstack()
ms.style.format('{:.1f}')

nb-dogs,1,2,3,4,6,10,15,20,25,35
nb-sheep,,,,,,,,,,
5,42.0,20.6,14.1,13.1,12.2,13.4,11.6,10.4,13.0,12.2
10,28.1,15.9,15.7,14.3,15.2,12.7,14.2,16.9,18.0,21.4
25,17.6,16.1,15.7,14.5,14.3,15.2,14.5,14.7,15.8,18.2
50,21.2,18.5,17.3,15.9,15.4,15.6,15.5,16.7,17.0,17.1
100,27.3,21.8,16.6,15.7,14.5,13.9,14.3,17.1,15.7,16.9
150,49.3,23.4,18.8,20.6,15.0,15.6,15.6,16.2,16.0,17.7
200,63.5,39.1,23.1,18.7,15.6,15.8,16.2,16.6,16.9,18.9
250,70.0,57.1,37.4,31.7,17.4,16.5,17.3,17.6,17.5,19.0
300,76.0,71.6,60.2,43.0,28.5,17.6,17.9,18.7,18.2,19.4


## 4. Total Dog Distance

In [6]:
dd = df.groupby(['nb-sheep','nb-dogs'])['dogs-distance'].mean().unstack()
dd.style.format('{:,.0f}')

nb-dogs,1,2,3,4,6,10,15,20,25,35
nb-sheep,,,,,,,,,,
5,"1,610","2,429","3,323","4,732","6,658","11,906","22,270","32,768","44,999","62,325"
10,"1,374","2,577","3,795","5,061","8,592","15,821","28,557","57,813","75,200","149,832"
25,"1,351","2,993","4,271","5,893","9,379","15,757","29,432","45,125","57,318","103,614"
50,"2,381","4,210","5,267","7,393","11,448","18,671","29,991","44,660","55,814","106,143"
100,"2,770","3,938","6,982","9,049","15,127","24,831","35,411","54,057","66,936","93,894"
150,"9,665","10,174","9,672","13,446","22,710","31,163","45,853","52,795","72,499","100,362"
200,"10,000","18,210","20,396","22,745","32,645","40,999","55,564","66,773","73,768","98,074"
250,"10,000","19,721","27,731","34,777","41,488","55,723","66,172","76,527","92,394","121,607"
300,"10,000","20,087","29,577","37,923","52,667","68,182","79,512","81,820","100,032","123,561"


## 5. Composite Predictor Search

In [7]:
cond = df.groupby(['nb-sheep','nb-dogs']).agg(
    sr=('success','mean'),
    ms=('mean-spreadness','mean'),
).reset_index()
cond['ratio'] = cond['nb-dogs'] / cond['nb-sheep']
cond['N_over_D'] = cond['nb-sheep'] / cond['nb-dogs']

def evaluate_rank(name, vals, sr):
    rho, p = stats.spearmanr(vals, sr)
    n_total = len(sr)
    best_correct, best_thr = 0, 0
    for thr in np.sort(vals.unique()):
        correct = ((vals <= thr) == (sr >= 0.5)).sum()
        if correct > best_correct:
            best_correct = correct
            best_thr = thr
    return {
        'Predictor': name,
        'Spearman ρ': rho,
        'Correct': best_correct,
        'Total': n_total,
        'Thr. acc.': best_correct / n_total,
        'Cutoff': best_thr,
    }

N = cond['nb-sheep']; D = cond['nb-dogs']; MS = cond['ms']; SR = cond['sr']

results_rank = pd.DataFrame([
    evaluate_rank('ms',              MS,                             SR),
    evaluate_rank('ms × √N',        MS * np.sqrt(N),                SR),
    evaluate_rank('ms × N',         MS * N,                         SR),
    evaluate_rank('ms × N/D',       MS * cond['N_over_D'],          SR),
]).set_index('Predictor')

results_rank.style.format({
    'Spearman ρ': '{:+.3f}',
    'Correct': '{:.0f}',
    'Total': '{:.0f}',
    'Thr. acc.': '{:.1%}',
    'Cutoff': '{:.0f}',
})

,Spearman ρ,Correct,Total,Thr. acc.,Cutoff
Predictor,,,,,
ms,-0.701,105,110,95.5%,28
ms × √N,-0.819,107,110,97.3%,417
ms × N,-0.828,105,110,95.5%,7026
ms × N/D,-0.614,104,110,94.5%,1752


Same search at two stricter operating thresholds: success rate < 15% (near-total failure) and >= 90% (reliable herding).

In [8]:
def evaluate_rank(name, vals, sr, success_threshold=0.5):
    rho, p = stats.spearmanr(vals, sr)
    n_total = len(sr)
    best_correct, best_thr = 0, 0
    for thr in np.sort(np.unique(vals)):
        correct = ((vals <= thr) == (sr >= success_threshold)).sum()
        if correct > best_correct:
            best_correct = correct
            best_thr = thr
    return {
        'Predictor': name,
        'Spearman ρ': rho,
        'Correct': best_correct,
        'Total': n_total,
        'Thr. acc.': best_correct / n_total,
        'Cutoff': best_thr,
    }

N = cond['nb-sheep'].values.astype(float)
D = cond['nb-dogs'].values.astype(float)
MS = cond['ms'].values
SR = cond['sr'].values

for threshold in [0.15, 0.90]:
    print(f"\n Success threshold: {threshold:.0%} ")

    preds = {
        'ms':        MS,
        'ms × √N':  MS * np.sqrt(N),
        'ms × N':   MS * N,
        'ms × N/D': MS * N / D,
    }

    results = pd.DataFrame([
        evaluate_rank(name, vals, SR, threshold)
        for name, vals in preds.items()
    ]).set_index('Predictor')
    print(results[['Spearman ρ', 'Correct', 'Total', 'Thr. acc.', 'Cutoff']].to_string())

    # Confusion matrix for best predictor
    best = results['Correct'].idxmax()
    vals = preds[best]
    cutoff = results.loc[best, 'Cutoff']

    predicted_positive = vals <= cutoff
    actual_positive = SR >= threshold

    tp = ( predicted_positive &  actual_positive).sum()
    fp = ( predicted_positive & ~actual_positive).sum()
    fn = (~predicted_positive &  actual_positive).sum()
    tn = (~predicted_positive & ~actual_positive).sum()

    print(f"\n  Best: {best} (cutoff = {cutoff:.1f})")
    print(f"  Confusion matrix (predict SR {'≥' if threshold >= 0.5 else '<'} {threshold:.0%}):")
    print(f"  {'':>20} {'Pred ≥':>10} {'Pred <':>10}")
    print(f"  {'Actual ≥ ' + f'{threshold:.0%}':>20} {tp:>10} {fn:>10}")
    print(f"  {'Actual < ' + f'{threshold:.0%}':>20} {fp:>10} {tn:>10}")
    print(f"  Precision: {tp/(tp+fp):.1%}  Recall: {tp/(tp+fn):.1%}  Accuracy: {(tp+tn)/len(SR):.1%}")


 Success threshold: 15% 
           Spearman ρ  Correct  Total  Thr. acc.       Cutoff
Predictor                                                    
ms          -0.701346      108    110   0.981818    28.082566
ms × √N     -0.818658      110    110   1.000000   416.583206
ms × N      -0.828321      107    110   0.972727  7816.412995
ms × N/D    -0.613763      107    110   0.972727  1752.436502

  Best: ms × √N (cutoff = 416.6)
  Confusion matrix (predict SR < 15%):
                           Pred ≥     Pred <
          Actual ≥ 15%         87          0
          Actual < 15%          0         23
  Precision: 100.0%  Recall: 100.0%  Accuracy: 100.0%

 Success threshold: 90% 
           Spearman ρ  Correct  Total  Thr. acc.       Cutoff
Predictor                                                    
ms          -0.701346       86    110   0.781818    17.069909
ms × √N     -0.818658       94    110   0.854545   252.312205
ms × N      -0.828321       96    110   0.872727  3090.180787
ms ×

## 6. False Positives and False Negatives

In [9]:
cutoff = 3090.180787 + 1e-6  # floating point
pred_vals = MS * N

# <= cutoff means "predict reliable (≥90%)"
predicted_reliable = pred_vals <= cutoff
actual_reliable = SR >= 0.90

fn_mask = actual_reliable & ~predicted_reliable
fp_mask = ~actual_reliable & predicted_reliable

# Verify totals match confusion matrix
tp = ( actual_reliable &  predicted_reliable).sum()
fp = fp_mask.sum()
fn = fn_mask.sum()
tn = (~actual_reliable & ~predicted_reliable).sum()
print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}, Total={tp+fp+fn+tn}\n")

print(f"{fn} False Negatives (actual ≥90% but ms×N > {cutoff:.1f})")
print(cond[fn_mask][['nb-sheep','nb-dogs','sr','ms']].assign(
    msN=pred_vals[fn_mask]).sort_values('nb-sheep').to_string(index=False))

print()

print(f"{fp} False Positives (actual <90% but ms×N ≤ {cutoff:.1f})")
print(cond[fp_mask][['nb-sheep','nb-dogs','sr','ms']].assign(
    msN=pred_vals[fp_mask]).sort_values('nb-sheep').to_string(index=False))

TP=53, FP=5, FN=9, TN=43, Total=110

9 False Negatives (actual ≥90% but ms×N > 3090.2)
 nb-sheep  nb-dogs   sr        ms         msN
      200       20 0.93 16.617020 3323.403935
      200       25 0.94 16.919025 3383.804966
      250       25 0.93 17.520903 4380.225652
      250       35 0.91 18.985909 4746.477352
      300       20 0.93 18.682554 5604.766290
      300       25 0.93 18.161792 5448.537711
      300       35 0.90 19.440667 5832.200013
      350       35 0.91 20.073105 7025.586611
      400       35 0.91 20.829160 8331.664128

5 False Positives (actual <90% but ms×N ≤ 3090.2)
 nb-sheep  nb-dogs   sr        ms        msN
       10       20 0.86 16.889904 168.899037
       10       25 0.85 18.037114 180.371137
       10       35 0.70 21.426844 214.268442
       25       35 0.88 18.211823 455.295581
       50       35 0.81 17.070162 853.508114


## 7. Failure Analysis

In [10]:
# --- Failure analysis by regime ---

all_failed = df[~df['success']]
n_fail = len(all_failed)
n_total = len(df)
print(f"Total: {n_fail} failures / {n_total} runs ({n_fail/n_total:.1%})\n")

print("Phase at timeout (all failures):")
for phase in ['collecting', 'holding', 'exiting']:
    count = (all_failed['current-phase'] == phase).sum()
    print(f"  {phase:<15}: {count:>5} ({count/n_fail:.0%})")
print()

# Split by regime
ctrl_keys = cond[cond['ms'] <= 25][['nb-sheep','nb-dogs']]
unctrl_keys = cond[cond['ms'] > 25][['nb-sheep','nb-dogs']]

for label, keys in [('ms > 25 (no control)', unctrl_keys),
                     ('ms ≤ 25 (controlled)', ctrl_keys)]:
    regime_df = df.merge(keys, on=['nb-sheep','nb-dogs'])
    failed = regime_df[~regime_df['success']]
    n_regime = len(regime_df)
    n_f = len(failed)

    print(f"{label}")
    print(f"  {n_f} failures / {n_regime} runs ({n_f/n_regime:.1%})")
    print(f"  {n_f/n_fail:.1%} of all failures")
    print()

    # Phase breakdown
    print(f"  Phase at timeout:")
    for phase in ['collecting', 'holding', 'exiting']:
        count = (failed['current-phase'] == phase).sum()
        pct = count / n_f if n_f > 0 else 0
        print(f"    {phase:<15}: {count:>5} ({pct:.0%})")
    print()

    # Captured sheep
    zero = (failed['sheep-captured-count'] == 0).sum()
    some = ((failed['sheep-captured-count'] > 0)).sum()
    mean_capt = failed['sheep-captured-count'].mean()
    print(f"  Captured sheep:")
    print(f"    Captured 0:      {zero:>5} ({zero/n_f:.0%})")
    print(f"    Captured ≥1:     {some:>5} ({some/n_f:.0%})")
    print(f"    Mean captured:   {mean_capt:.1f}")
    print()


Total: 3089 failures / 11000 runs (28.1%)

Phase at timeout (all failures):
  collecting     :  2787 (90%)
  holding        :    93 (3%)
  exiting        :   209 (7%)

ms > 25 (no control)
  2161 failures / 2500 runs (86.4%)
  70.0% of all failures

  Phase at timeout:
    collecting     :  2104 (97%)
    holding        :    14 (1%)
    exiting        :    43 (2%)

  Captured sheep:
    Captured 0:       2143 (99%)
    Captured ≥1:        18 (1%)
    Mean captured:   0.8

ms ≤ 25 (controlled)
  928 failures / 8500 runs (10.9%)
  30.0% of all failures

  Phase at timeout:
    collecting     :   683 (74%)
    holding        :    79 (9%)
    exiting        :   166 (18%)

  Captured sheep:
    Captured 0:        863 (93%)
    Captured ≥1:        65 (7%)
    Mean captured:   8.8



## 8. Minimum Dogs for Reliable Herding

In [11]:
sr_pivot = cond.pivot(index='nb-sheep', columns='nb-dogs', values='sr') * 100

for threshold in [90, 75, 50]:
    print(f"Threshold: {threshold}%")
    for N_val in sr_pivot.index:
        rates = sr_pivot.loc[N_val].dropna().sort_index()
        viable = rates[rates >= threshold]
        if len(viable) > 0:
            D_min = viable.index[0]
            print(f"  N={N_val:>3}: D_min={D_min:>2} (1:{N_val//D_min}, SR={viable.iloc[0]:.0f}%)")
        else:
            print(f"  N={N_val:>3}: no config reaches {threshold}%")
    print()

Threshold: 90%
  N=  5: D_min= 1 (1:5, SR=100%)
  N= 10: D_min= 1 (1:10, SR=99%)
  N= 25: D_min= 1 (1:25, SR=100%)
  N= 50: D_min= 1 (1:50, SR=100%)
  N=100: D_min= 1 (1:100, SR=100%)
  N=150: D_min= 3 (1:50, SR=99%)
  N=200: D_min=20 (1:10, SR=93%)
  N=250: D_min=25 (1:10, SR=93%)
  N=300: D_min=20 (1:15, SR=93%)
  N=350: D_min=35 (1:10, SR=91%)
  N=400: D_min=35 (1:11, SR=91%)

Threshold: 75%
  N=  5: D_min= 1 (1:5, SR=100%)
  N= 10: D_min= 1 (1:10, SR=99%)
  N= 25: D_min= 1 (1:25, SR=100%)
  N= 50: D_min= 1 (1:50, SR=100%)
  N=100: D_min= 1 (1:100, SR=100%)
  N=150: D_min= 2 (1:75, SR=80%)
  N=200: D_min=10 (1:20, SR=89%)
  N=250: D_min=15 (1:16, SR=85%)
  N=300: D_min=20 (1:15, SR=93%)
  N=350: D_min=20 (1:17, SR=85%)
  N=400: D_min=25 (1:16, SR=85%)

Threshold: 50%
  N=  5: D_min= 1 (1:5, SR=100%)
  N= 10: D_min= 1 (1:10, SR=99%)
  N= 25: D_min= 1 (1:25, SR=100%)
  N= 50: D_min= 1 (1:50, SR=100%)
  N=100: D_min= 1 (1:100, SR=100%)
  N=150: D_min= 2 (1:75, SR=80%)
  N=200: D_min= 3

## 9. Crowding Effect

In [12]:
# For each N: find optimal D and effective range

print(f"{'N':>4} {'D_opt':>6} {'SR_max':>7} {'D range ≥90%':>15}")
print("-" * 38)

for N_val in sorted(cond['nb-sheep'].unique()):
    sub = cond[cond['nb-sheep'] == N_val].sort_values('nb-dogs')

    best = sub.loc[sub['sr'].idxmax()]
    D_opt = int(best['nb-dogs'])
    SR_max = best['sr']

    viable = sub[sub['sr'] >= 0.90]
    if len(viable) > 0:
        D_range = f"{int(viable['nb-dogs'].min())}-{int(viable['nb-dogs'].max())}"
    else:
        D_range = "none"

    print(f"{N_val:>4} {D_opt:>6} {SR_max:>6.0%} {D_range:>15}")

   N  D_opt  SR_max    D range ≥90%
--------------------------------------
   5      1   100%            1-35
  10      2   100%            1-15
  25      1   100%            1-25
  50      1   100%            1-25
 100      1   100%            1-35
 150      3    99%            3-35
 200     25    94%           20-25
 250     25    93%           25-35
 300     20    93%           20-35
 350     35    91%           35-35
 400     35    91%           35-35
